# Prompt Comparison — LLM Judge Evaluation

Evaluate whether baseline and modified messages make sense given conversation context.

In [ ]:
import sys
sys.path.insert(0, "../..")

import importlib
import helpers
importlib.reload(helpers)

import pandas as pd
from helpers import Helpers

h = Helpers(cache_dir="../../cache/prompt_comparison")

## Generate baseline / modified / comparison CSVs

Runs the shootout eval against `cases/write_for_me/` (both v1 and v2) twice:

- **Baseline** — prompts copied from `master:enki/priv/llm/prompts/write_for_me/` into the eval's prompt dir.
- **Modified** — prompts copied from the current working tree at `enki/priv/llm/prompts/write_for_me/`.

Both runs overwrite `rails/eval/prompts/write_for_me/*.md` in place, then restore it via `git checkout` after the eval finishes. The cells abort if that directory has uncommitted changes, to avoid clobbering in-progress work.

Skip this section and jump straight to the cell below if you already have a `comparison.csv`.

In [ ]:
import subprocess
from pathlib import Path

REPO_ROOT = Path("/Users/garrettsmith/RubymineProjects/secondary-aweb")
EVAL_DIR = REPO_ROOT / "rails/eval"
CASES_SUBDIR = "cases/write_for_me/"  # runs both v1 and v2
SHOOTOUT_MODEL = "claude-haiku-4-5"
PARALLELIZATION = 128  # eval framework default

EVAL_PROMPTS_DIR = EVAL_DIR / "prompts/write_for_me"
SOURCE_PROMPTS_REL = "enki/priv/llm/prompts/write_for_me"
SOURCE_PROMPTS_DIR = REPO_ROOT / SOURCE_PROMPTS_REL

BASELINE_CSV = EVAL_DIR / "baseline.csv"
MODIFIED_CSV = EVAL_DIR / "modified.csv"
COMPARISON_CSV = EVAL_DIR / "comparison.csv"


def _sh(cmd, cwd=None, check=True):
    r = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
    )
    if check and r.returncode != 0:
        raise RuntimeError(
            f"command failed ({r.returncode}): {cmd}\n"
            f"stdout:\n{r.stdout}\nstderr:\n{r.stderr}"
        )
    return r


def _assert_eval_prompts_clean():
    r = _sh(
        ["git", "-C", str(REPO_ROOT), "status", "--short", "--",
         "rails/eval/prompts/write_for_me"],
    )
    if r.stdout.strip():
        raise RuntimeError(
            "rails/eval/prompts/write_for_me has uncommitted changes; "
            "commit or stash them before running the eval cells:\n" + r.stdout
        )


def _stage_prompts_from_ref(ref):
    _assert_eval_prompts_clean()
    for p in EVAL_PROMPTS_DIR.glob("*.md"):
        rel = f"{SOURCE_PROMPTS_REL}/{p.name}"
        r = _sh(["git", "-C", str(REPO_ROOT), "show", f"{ref}:{rel}"], check=False)
        if r.returncode == 0:
            p.write_text(r.stdout)


def _stage_prompts_from_working_tree():
    _assert_eval_prompts_clean()
    for p in EVAL_PROMPTS_DIR.glob("*.md"):
        src = SOURCE_PROMPTS_DIR / p.name
        if src.exists():
            p.write_text(src.read_text())


def _restore_eval_prompts():
    _sh(
        ["git", "-C", str(REPO_ROOT), "checkout", "--",
         "rails/eval/prompts/write_for_me"],
    )


def run_shootout(out_csv: Path, stage_fn):
    """Stage prompts, run the shootout (streaming output), then restore prompts."""
    stage_fn()
    try:
        proc = subprocess.Popen(
            ["./bin/eval", "--shootout", "-m", SHOOTOUT_MODEL,
             "-p", str(PARALLELIZATION),
             "--csv", str(out_csv), CASES_SUBDIR],
            cwd=str(EVAL_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f"eval failed with exit code {rc}")
    finally:
        _restore_eval_prompts()

    lines = out_csv.read_text().splitlines()
    data_rows = max(len(lines) - 1, 0)
    print(f"\nwrote {out_csv} — {data_rows} data rows")
    if data_rows == 0:
        raise RuntimeError(
            f"{out_csv} is empty (header only). All cases likely errored."
        )

In [ ]:
# Baseline run: stage master's enki prompts, run shootout
run_shootout(BASELINE_CSV, lambda: _stage_prompts_from_ref("master"))

In [ ]:
# Modified run: stage current working-tree enki prompts, run shootout
run_shootout(MODIFIED_CSV, _stage_prompts_from_working_tree)

In [ ]:
import json
import yaml
from pathlib import Path

def _read_shootout_csv(path):
    d = pd.read_csv(path)
    return d.drop_duplicates(subset="case_name", keep="first")[
        ["case_name", "file_path", "generated_message"]
    ]

def _case_context(row):
    yaml_path = Path(row["file_path"])
    if not yaml_path.is_absolute():
        yaml_path = EVAL_DIR / yaml_path
    meta = yaml.safe_load(yaml_path.read_text())
    s3_key = (meta.get("s3_input") or {}).get("key")
    if s3_key:
        cache_json = EVAL_DIR / "tmp/s3_cache" / s3_key
        if cache_json.exists():
            return cache_json.read_text()
        raise FileNotFoundError(
            f"S3 cache miss for {s3_key}. Run `./bin/eval-data sync` from {EVAL_DIR}."
        )
    return json.dumps(meta.get("input", {}), indent=2)

def _prompt_slug(file_path: str) -> str:
    # The variant is the immediate parent dir of the case yml, regardless of nesting.
    # cases/write_for_me/v1/general/xxx.yml               → write_for_me/general
    # cases/write_for_me/v2/none/none_no_contact/xxx.yml  → write_for_me/none_no_contact
    parts = Path(file_path).parts
    return f"{parts[1]}/{parts[-2]}"

baseline = _read_shootout_csv(BASELINE_CSV).rename(columns={"generated_message": "baseline_message"})
modified = _read_shootout_csv(MODIFIED_CSV).rename(columns={"generated_message": "modified_message"})

merged = baseline.merge(modified[["case_name", "modified_message"]], on="case_name", how="inner")
merged["prompt"] = merged["file_path"].map(_prompt_slug)
merged["context"] = merged.apply(_case_context, axis=1)

merged = merged[["case_name", "prompt", "baseline_message", "modified_message", "context"]]
merged.to_csv(COMPARISON_CSV, index=False)
print(f"comparison.csv → {COMPARISON_CSV} ({len(merged)} rows)")

## Per-rubric-question analysis

Runs `./bin/analyze-shootout --by-question` against the combined baseline + modified shootout CSVs, then computes a per-question pass-rate delta. Surfaces the rubric questions where the modified prompts regressed (or improved) most vs. baseline.

This is independent of the LLM judge — it uses the rubric-pass data the shootout already produced.

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd

# Combine baseline + modified into one CSV with relabeled model column,
# so analyze-shootout can produce a side-by-side per-question comparison.
# IMPORTANT: read with dtype=str + keep_default_na=False so pandas doesn't
# convert "true"/"false" → True/False (which serialize back as "True"/"False"
# and break the Ruby script's case-sensitive "true" comparison → all 0%).
_combined = EVAL_DIR / "tmp" / "combined_for_analysis.csv"
_per_question = EVAL_DIR / "tmp" / "per_question_analysis.csv"
_combined.parent.mkdir(parents=True, exist_ok=True)

_b = pd.read_csv(BASELINE_CSV, dtype=str, keep_default_na=False)
_m = pd.read_csv(MODIFIED_CSV, dtype=str, keep_default_na=False)
_b["model"] = "baseline"
_m["model"] = "modified"
pd.concat([_b, _m], ignore_index=True).to_csv(_combined, index=False)

# Run analyze-shootout --by-question --export-csv
proc = subprocess.run(
    ["./bin/analyze-shootout", "--by-question",
     "--export-csv", str(_per_question),
     str(_combined)],
    cwd=str(EVAL_DIR),
    capture_output=True, text=True,
)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr)
    raise RuntimeError(f"analyze-shootout failed (exit {proc.returncode})")

# Load the per-question pass-rate CSV (columns: question, baseline, modified)
pq_df = pd.read_csv(_per_question)

# Split into:
#   - compared:   questions that fired in BOTH runs (real apples-to-apples comparison)
#   - asymmetric: questions that fired in only one run (rubric conditional logic
#                 differed between baseline and modified — can't compute a delta)
both_present = pq_df["baseline"].notna() & pq_df["modified"].notna()
compared = pq_df[both_present].copy()
asymmetric = pq_df[~both_present].copy()

compared["delta"] = compared["modified"] - compared["baseline"]
compared_sorted = compared.sort_values("delta", ascending=True).reset_index(drop=True)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

print(f"\n{len(pq_df)} total rubric questions in combined data.")
print(f"  Comparable (fired in both runs):     {len(compared)}")
print(f"  Asymmetric (fired in only one run):  {len(asymmetric)}")
print()
print(f"Of the {len(compared)} comparable questions:")
print(f"  Regressions (delta < 0):  {(compared['delta'] < 0).sum()}")
print(f"  Improvements (delta > 0): {(compared['delta'] > 0).sum()}")
print(f"  Unchanged (delta == 0):   {(compared['delta'] == 0).sum()}")

print("\n" + "="*100)
print("Top 20 regressions (modified pass rate dropped most vs baseline):")
print("="*100)
regressions = compared_sorted[compared_sorted["delta"] < 0].head(20)
display(regressions if not regressions.empty else "No regressions.")

print("\n" + "="*100)
print("Top 20 improvements (modified pass rate jumped most vs baseline):")
print("="*100)
improvements = compared_sorted[compared_sorted["delta"] > 0].tail(20).iloc[::-1].reset_index(drop=True)
display(improvements if not improvements.empty else "No improvements.")

if len(asymmetric) > 0:
    print("\n" + "="*100)
    print(f"Asymmetric questions ({len(asymmetric)} — fired in only one run, no delta possible):")
    print("="*100)
    print("These are typically conditional rubric questions whose triggering classifier")
    print("fired differently between the two runs. They're informational, not a comparison.")
    display(asymmetric.reset_index(drop=True))

In [ ]:
regressions[regressions["delta"] < -5.0].to_clipboard(sep=',')
# regressions[regressions["delta"] < -5.0].to_csv("prompt_update_evals_without_temp_change.csv", index=False)

In [ ]:
df = pd.read_csv("/Users/garrettsmith/RubymineProjects/secondary-aweb/rails/eval/comparison.csv")
print(f"{len(df):,} rows")
df.head(2)

## Cost Estimate

In [ ]:
h.estimate_judge_cost(
    df["baseline_message"].tolist(),
    df["modified_message"].tolist(),
    df["context"].tolist(),
    model=h.OPUS_PROFILE,
)

## Run LLM Judge

In [ ]:
verdicts = h.judge_parallel(
    df["baseline_message"].tolist(),
    df["modified_message"].tolist(),
    df["context"].tolist(),
    model=h.OPUS_PROFILE,
)

df["baseline_makes_sense"] = [v["baseline_makes_sense"] for v in verdicts]
df["baseline_reasoning"] = [v["baseline_reasoning"] for v in verdicts]
df["modified_makes_sense"] = [v["modified_makes_sense"] for v in verdicts]
df["modified_reasoning"] = [v["modified_reasoning"] for v in verdicts]
df["comparison_notes"] = [v.get("comparison_notes", "") for v in verdicts]

## Results

In [ ]:
print("Baseline makes sense:")
print(df["baseline_makes_sense"].value_counts())
print(f"\nModified makes sense:")
print(df["modified_makes_sense"].value_counts())

# Cases where they disagree
disagree = df[df["baseline_makes_sense"] != df["modified_makes_sense"]]
print(f"\nDisagreements (baseline vs modified): {len(disagree):,} / {len(df):,}")

df[["case_name", "baseline_makes_sense", "baseline_reasoning", "modified_makes_sense", "modified_reasoning", "comparison_notes"]].head(10)

## Comparison: Quality + Meeting Likelihood

In [ ]:
h.estimate_compare_cost(
    df["baseline_message"].tolist(),
    df["modified_message"].tolist(),
    df["context"].tolist(),
    model=h.OPUS_PROFILE,
)

In [ ]:
comparisons = h.compare_parallel(
    df["baseline_message"].tolist(),
    df["modified_message"].tolist(),
    df["context"].tolist(),
    model=h.OPUS_PROFILE,
)

df["quality_verdict"] = [c["quality_verdict"] for c in comparisons]
df["quality_reasoning"] = [c["quality_reasoning"] for c in comparisons]
df["meeting_verdict"] = [c["meeting_verdict"] for c in comparisons]
df["meeting_reasoning"] = [c["meeting_reasoning"] for c in comparisons]

In [ ]:
print("Quality verdict (is modified better or worse?):")
print(df["quality_verdict"].value_counts())

print(f"\nMeeting likelihood verdict:")
print(df["meeting_verdict"].value_counts())

df[["case_name", "quality_verdict", "quality_reasoning", "meeting_verdict", "meeting_reasoning"]].head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- 1. Quality verdict bar chart ---
ax = axes[0]
quality_counts = df["quality_verdict"].value_counts()
colors = {"Better": "#4CAF50", "Worse": "#F44336"}
bars = ax.bar(quality_counts.index, quality_counts.values,
              color=[colors.get(v, "#999") for v in quality_counts.index])
ax.set_title("Is the Modified Message Better or Worse?", fontweight="bold")
ax.set_ylabel("Count")
for bar, val in zip(bars, quality_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{val} ({val/len(df):.0%})", ha="center", fontweight="bold")

# --- 2. Meeting likelihood bar chart ---
ax = axes[1]
meeting_counts = df["meeting_verdict"].value_counts().reindex(["Modified", "Baseline", "N/A"]).dropna().astype(int)
colors_m = {"Modified": "#4CAF50", "Baseline": "#2196F3", "N/A": "#9E9E9E"}
bars = ax.bar(meeting_counts.index, meeting_counts.values,
              color=[colors_m.get(v, "#999") for v in meeting_counts.index])
ax.set_title("Which Message More Likely to Generate a Meeting?", fontweight="bold")
ax.set_ylabel("Count")
for bar, val in zip(bars, meeting_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{val} ({val/len(df):.0%})", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## Per-prompt regression analysis

Two signals that the modified prompt did *worse* than baseline:
- `quality_verdict == "Worse"` — judge rated the modified message lower-quality than the baseline
- `meeting_verdict == "Baseline"` — judge picked the baseline as more likely to produce a meeting

A row is "bad" if either signal fires. Aggregating by `prompt` shows which prompt files regressed the most. The bottom section lists individual cases where *both* signals fire — those are the highest-priority cases to inspect.

In [ ]:
is_worse = df["quality_verdict"] == "Worse"
is_baseline_meeting = df["meeting_verdict"] == "Baseline"
is_bad = is_worse | is_baseline_meeting

agg = df.assign(
    _worse=is_worse,
    _baseline_meeting=is_baseline_meeting,
    _bad=is_bad,
).groupby("prompt").agg(
    n=("case_name", "count"),
    n_worse=("_worse", "sum"),
    n_baseline_meeting=("_baseline_meeting", "sum"),
    n_bad=("_bad", "sum"),
)
agg["pct_worse"] = (agg["n_worse"] / agg["n"] * 100).round(1)
agg["pct_baseline_meeting"] = (agg["n_baseline_meeting"] / agg["n"] * 100).round(1)
agg["pct_bad"] = (agg["n_bad"] / agg["n"] * 100).round(1)
agg = agg.sort_values("pct_bad", ascending=False)

print("Per-prompt regression rates (sorted by combined bad rate):")
print(agg[["n", "n_worse", "pct_worse", "n_baseline_meeting", "pct_baseline_meeting", "n_bad", "pct_bad"]].to_string())

print()
print(f"Cases where BOTH signals fire (modified worse AND baseline wins meeting): "
      f"{(is_worse & is_baseline_meeting).sum()} / {len(df)}")
print()
print("Worst individual cases (both signals fire):")
worst = df[is_worse & is_baseline_meeting][
    ["prompt", "case_name", "quality_reasoning", "meeting_reasoning"]
]
for _, row in worst.head(20).iterrows():
    print(f"\n• [{row['prompt']}] {row['case_name']}")
    print(f"    quality: {row['quality_reasoning'][:200]}")
    print(f"    meeting: {row['meeting_reasoning'][:200]}")

In [ ]:
df.to_csv("prompt_comparison_results.csv", index=False)
print(f"Exported {len(df):,} rows to prompt_comparison_results.csv")